## SummaryThis notebook quantifies persistency from the bi-modal milk flow data.It tracks how milk yield is maintained after peak and evaluates treatment-related effects across the lactation curve.

# Bi-modal milk flow project

Analysis of persistency based on MilkBot LC model frequentist fitted. Persistency is calculated as average slope of the decay phase of the LC until day 305. 

In [0]:
%pip install --upgrade shap "numpy<=2.1"
%restart_python
import shap
shap.initjs()
print(shap.__version__)

In [0]:
#import packages 
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import seaborn as sns
# from statsmodels.formula.api import ols
from scipy.stats import norm 
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit

Load data from blob 

In [0]:
#read dataframe from the US blob 
parquet_file_path = "wasbs://bovianalytics-data@bovianalyticsstorage.blob.core.windows.net/us/SunnySide/lactation_curve_characteristics_half_time_persistency"

# Read the Parquet file into a DataFrame > now set to a limit to first make the code working
bi_modal_df = spark.read.parquet(parquet_file_path)

In [0]:
print('Mean persistency', bi_modal_df.select(F.avg('Persistency')).collect()[0][0])

#also calculate median 
print('Median persistency', bi_modal_df.approxQuantile('Persistency', [0.5], 0.01)[0])

In [0]:
# make a list of lactation ID with LactationCat = 1 and persistency > 6700
high_persistency = bi_modal_df.filter((bi_modal_df['LactationNumber'] == 1) & (bi_modal_df['Persistency'] > 6700)).select('LactationId').toPandas()['LactationId'].tolist()
print(high_persistency)

In [0]:
display(bi_modal_df)

Databricks visualization. Run in Databricks to view.

In [0]:
#convert to panda to make data easier to plot
pdf = bi_modal_df.toPandas()

In [0]:
#count lactations where cummulativeyield < 0 
below_zero =  pdf[pdf['CummulativeYield'] < 0]

print("Number of lactations with negative yield: ", below_zero.shape[0])

In [0]:
#some mismatched fitting
#drop negative milk yield 
pdf = pdf[pdf['CummulativeYield'] > 0]

# time to peak > 5 & < 305
pdf = pdf[pdf['TimeToPeak'] < 305]
pdf = pdf[pdf['TimeToPeak'] > 5]

#peak yield > 127 kg
pdf = pdf[pdf['PeakYield'] < 127]



In [0]:
display(pdf)

In [0]:
#visualize model variables 
plt.hist(
    pdf['TimeToPeak'],
    bins=100
)
plt.title('Time to peak distribution')
plt.xlabel('Time to peak')
plt.ylabel('Frequency')
plt.show()

plt.hist(pdf['PeakYield'], bins=100)
plt.title('Peak distribution')
plt.xlabel('Peak')
plt.ylabel('Frequency')
plt.show()

plt.hist(pdf['CummulativeYield'], bins=100)
plt.title('Cummulative yield distribution')
plt.xlabel('Cummulative yield')
plt.ylabel('Frequency')
plt.show()

plt.hist(pdf['Persistency'], bins=100)
plt.title('Persistency distribution')
plt.xlabel('Persistency')
plt.ylabel('Frequency')
plt.show()

plt.hist(pdf['DailyMilkYield'], bins = 100)
plt.title('Daily milk yield distribution')
plt.xlabel('Daily milk yield')
plt.ylabel('Frequency')
plt.show

In [0]:
#visualize model variables 
# plt.hist(pdf['TimeToPeak'], bins=100)
# plt.title('Time to peak distribution')
# plt.xlabel('Time to peak')
# plt.ylabel('Frequency')
# plt.show()

# plt.hist(pdf['PeakYield'], bins=100)
# plt.title('Peak distribution')
# plt.xlabel('Peak')
# plt.ylabel('Frequency')
# plt.show()

# plt.hist(pdf['CummulativeYield'], bins=100)
# plt.title('Cummulative yield distribution')
# plt.xlabel('Cummulative yield')
# plt.ylabel('Frequency')
# plt.show()

plt.hist(pdf['Persistency'], bins=100)
plt.title('Persistency distribution')
plt.xlabel('Persistency')
plt.ylabel('Frequency')
plt.show()

# plt.hist(pdf['DailyMilkYield'], bins = 100)
# plt.title('Daily milk yield distribution')
# plt.xlabel('Daily milk yield')
# plt.ylabel('Frequency')
# plt.show

Take log of persistency

### Create categorial variable for DME 

In [0]:
#alternative create a categorial variable out of BMI prop 
#rename column BIM to DME 
pdf.rename(columns={'BIM_prop': 'DME_prop'}, inplace=True)
pdf.rename(columns ={'BIM_prop_logit': 'DME_prop_logit'}, inplace=True)

# #equal split in three
# pdf["DME_prop_cat"] = pd.qcut(pdf["DME_prop"], q=3, labels=["low", "middle", "high"])

#split based on a threshold 
bins = [0, 0.20, 0.40, 1.0]
labels = ["low", "middle", "high"]

pdf["DME_prop_cat"] = pd.cut(pdf["DME_prop"], bins=bins, labels=labels, include_lowest=True)


### Create categorial variable for lactation number

In [0]:
#convert lactationnumber to 4 categories, 1,2,3,4 where everything above 4 becomes 4. 
pdf["LactationCat"] = pdf["LactationNumber"].clip(upper=4)

### Descriptive stats persistency

In [0]:
#plot Persistency vs DME prop and vs Logit DME

plt.scatter(pdf['Persistency'], pdf['DME_prop'])
# add trendline 
lr = LinearRegression()
lr.fit(pdf['Persistency'].values.reshape(-1, 1), pdf['DME_prop'])
plt.plot(pdf['Persistency'], lr.predict(pdf['Persistency'].values.reshape(-1, 1)), color='red')
plt.title('Persistency vs DME prop')
plt.xlabel('Persistency')
plt.ylabel('DME prop')
plt.show()  

plt.scatter(pdf['Persistency'], pdf['DME_prop_logit'])
# add trendline 
lr = LinearRegression()
lr.fit(pdf['Persistency'].values.reshape(-1, 1), pdf['DME_prop_logit'])
plt.plot(pdf['Persistency'], lr.predict(pdf['Persistency'].values.reshape(-1, 1)), color='red')
plt.title('Persistency vs Logit DME')
plt.xlabel('Persistency')
plt.ylabel('Logit BMI')
plt.show()


In [0]:
#plot Persistency vs LactationNumber, CalvingSeason and CalvingYear 

plt.scatter(pdf['Persistency'], pdf['LactationNumber'])
# add trendline 
lr = LinearRegression()
lr.fit(pdf['Persistency'].values.reshape(-1, 1), pdf['LactationNumber'])
plt.plot(pdf['Persistency'], lr.predict(pdf['Persistency'].values.reshape(-1, 1)), color='red')
plt.title('Persistency vs lactation number')
plt.xlabel('Persistency')
plt.ylabel('Lactation number')
plt.show()  

# Boxplot: Persistency by CalvingSeason
data = [pdf.loc[pdf['CalvingSeason'] == season, 'Persistency'] for season in sorted(pdf['CalvingSeason'].unique())]
plt.boxplot(data, labels=sorted(pdf['CalvingSeason'].unique()))
plt.title('Persistency vs CalvingSeason')
plt.xlabel('CalvingSeason')
plt.ylabel('Persistency')
plt.show()

# Boxplot: Persistency by CalvingYear
data = [
    pdf.loc[pdf['CalvingYear'] == year, 'Persistency']
    for year in sorted(pdf['CalvingYear'].unique())
]
plt.boxplot(data, labels=sorted(pdf['CalvingYear'].unique()))
plt.title('Persistency vs CalvingYear')
plt.xlabel('CalvingYear')
plt.ylabel('Persistency')
plt.show()

In [0]:
pdf['PeakYield'].max()

In [0]:
#Log transform peristency
pdf['logPersistency'] = np.log(pdf['Persistency'])

#### Correlations

In [0]:
#test if predictors are correlated 
# Convert CalvingSeason to numeric codes if it is categorical
pdf['CalvingSeason_cat'] = pdf['CalvingSeason'].astype('category').cat.codes

# Compute correlation matrix
corr_matrix = pdf[
    ['DME_prop_logit', 'CalvingSeason_cat', 'LactationCat', 'CalvingYear' , 'Persistency', 'logPersistency']
].corr()

sns.heatmap(
    corr_matrix, 
    annot=True, 
    cmap='coolwarm'
)
plt.title('Correlation Matrix')
plt.show()

Also shows neg correlation between DME_prop_logit and lactationnumber 
pos correlation between DME and persistency 

In [0]:
#show different CalvingYear
pdf.groupby('CalvingYear').size()

## Train-test split

In [0]:
#perform a train test split of the dataset based on AnimalNumber
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(pdf, groups=pdf["AnimalNumber"])
)

train_df = pdf.iloc[train_idx]
test_df = pdf.iloc[test_idx] #now contains 21.2 % of the data 
#for the train set
#convert calvingseason by one hot encoding 
train_df = pd.get_dummies(train_df, columns=['CalvingSeason'], prefix = 'CalvingSeason', dtype=int)
#rename columns so they match again with the seasons 
train_df.rename(columns={'CalvingSeason_0': 'CalvingSeason_Spring', 'CalvingSeason_1': 'CalvingSeason_Summer', 'CalvingSeason_2': 'CalvingSeason_Autumn', 'CalvingSeason_3': 'CalvingSeason_Winter'}, inplace=True)


#mean calvingyear train set 
mean_calving_year = train_df['CalvingYear'].mean()
#center calvingyear 
train_df['CalvingYear'] = train_df['CalvingYear'] - mean_calving_year


#for the test set 
#convert calvingseason by one hot encoding 
test_df = pd.get_dummies(test_df, columns=['CalvingSeason'], prefix = 'CalvingSeason', dtype=int)
#rename columns so they match again with the seasons 
test_df.rename(columns={'CalvingSeason_0': 'CalvingSeason_Spring', 'CalvingSeason_1': 'CalvingSeason_Summer', 'CalvingSeason_2': 'CalvingSeason_Autumn', 'CalvingSeason_3': 'CalvingSeason_Winter'}, inplace=True)
#center calvingyear using the mean of the train set 
test_df['CalvingYear'] = test_df['CalvingYear'] - mean_calving_year

## Linear Regression models

### Classic

In [0]:
model_305 = ols(
    'logPersistency ~ DME_prop_logit + C(LactationCat) + CalvingYear + CalvingSeason_Spring + CalvingSeason_Summer + CalvingSeason_Autumn',
    data=train_df
).fit()


print(model_305.summary())

# Predictions
y = train_df['logPersistency'].values
yhat = model_305.fittedvalues

# Visualization
plt.figure(figsize=(6,6))
plt.scatter(y, yhat, alpha=0.5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], linewidth=2)

plt.xlabel("Observed logPersistency")
plt.ylabel("Fitted logPersistency")
plt.title("Observed vs Fitted")
plt.tight_layout()
plt.show()

# RMSE
print('Train RMSE: ', np.sqrt(mean_squared_error(y, yhat)))

In [0]:
# Create visualizations to check model assumptions
plt.figure(figsize=(12, 6))

# qq-plot to see if residuals are normally distributed
plt.subplot(1, 2, 1)

residuals = model_305.resid
# Step 1: Fit a normal distribution to the residuals
mu, sigma = norm.fit(residuals)

# Step 2: Sort the residuals
sorted_residuals = np.sort(residuals)

# Step 3: Compute the theoretical quantiles from the normal distribution
n = len(residuals)
# create equally spaced probabilities (avoiding 0 and 1)
probabilities = (np.arange(1, n + 1) - 0.5) / n
theoretical_quantiles = norm.ppf(probabilities, loc=mu, scale=sigma)

# Step 4: Plot
plt.scatter(theoretical_quantiles, sorted_residuals, color='blue', label='Residuals')
plt.plot(theoretical_quantiles, theoretical_quantiles, color='red', lw=2, label='45° line')
plt.xlabel('Theoretical Quantiles (Normal)')
plt.ylabel('Sample Quantiles (Residuals)')
plt.title('QQ plot of residuals')
plt.legend()


# residuals vs fitted values to check for homoscedasticity
plt.subplot(1, 2, 2)
plt.scatter(yhat, model_305.resid)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Fitted values')
plt.ylabel('Residuals')
plt.title('Residuals vs Fitted values')

plt.tight_layout()
plt.show()

#### Coverting LactationNumber to categorial variables

In [0]:
# fit linear regression model 

#drop winter 
model_305_cont = ols('PeakYield ~ DME_prop_logit + C(LactationCat) + CalvingYear + CalvingSeason_Spring + CalvingSeason_Summer + CalvingSeason_Autumn', data=train_df)
model_305_cont = model_305_cont.fit()
print(model_305_cont.summary())

#visualize results
y = train_df['PeakYield'].values
yhat = model_305_cont.fittedvalues

plt.figure(figsize=(6,6))
plt.scatter(y, yhat, alpha=0.5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], linewidth=2)  # 1:1 line

plt.xlabel("Observed Peak milk yield")
plt.ylabel("Fitted Peak milk yield")
plt.title("Observed vs Fitted")
plt.tight_layout()
plt.show()

print('Train RMSE: ', np.sqrt(mean_squared_error(y, yhat)))

In [0]:
# Inverse logit function
def logit(x, eps=1e-6):
    return np.log((x + eps) / (1 - x + eps))

# Range of proportions
DME_vals = np.linspace(0, 1, 100)

# Coefficient for DME_prop_logit
beta = model_305.params['DME_prop_logit']

# Δ milk yield relative to DME_prop = 0
delta_yield = beta * (logit(DME_vals) - logit(0.000001))

plt.figure(figsize=(8, 5))
plt.plot(DME_vals, delta_yield)
plt.xlabel("DME proportion")
plt.ylabel("Δ persistency relative to no DME")
plt.title("Effect of DME proportion on LogPersistency")
plt.grid(True)
plt.show()

Transforming Logit(BMI) back to BMI 

In [0]:
import numpy as np

beta = model_305.params["DME_prop_logit"]

def percent_change_1pct(p):
    """
    Computes percent change in persistency
    for a +1 percentage point increase in DME_prop
    starting from proportion p.
    """

    eps = 1e-6

    def logit(x):
        return np.log((x + eps) / (1 - x + eps))

    # change in logit after +1 percentage point
    delta_logit = logit(p + 0.01) - logit(p)

    # change on log outcome scale
    delta_log_y = beta * delta_logit

    # convert back to percent change
    pct_change = (np.exp(delta_log_y) - 1) * 100

    return pct_change

# Examples
for p in [0.05, 0.10, 0.20, 0.30, 0.40]:
    effect = percent_change_1pct(p)

    print(
        f"At DME_prop={p:.2f}, "
        f"increasing DME_prop by 1 percentage point "
        f"changes expected persistency by {effect:.2f}%"
    )

# Effect at mean
mean_p = train_df["DME_prop"].mean()

effect_mean = percent_change_1pct(mean_p)

print(
    f"\nAt mean DME_prop ({mean_p:.3f}), "
    f"+1 percentage point changes expected persistency by "
    f"{effect_mean:.2f}%"
)


##### Eval test set

In [0]:
#Evaluate using the testset 

# Use the full test_df and let the model handle categorical encoding
y_test = test_df['logPersistency'].values
ypred_test = model_305.predict(test_df)
print('Test R2: ', r2_score(y_test, ypred_test))
print('Test RMSE: ', np.sqrt(mean_squared_error(y_test, ypred_test)))

#visualize results

plt.figure(figsize=(6,6))
plt.scatter(y_test, ypred_test, alpha=0.5, color = 'darkorange')
plt.plot([y.min(), y.max()], [y.min(), y.max()], linewidth=2, color = 'darkorange')  # 1:1 line

plt.xlabel("Observed logPersistency")
plt.ylabel("Fitted logPersistency")
plt.title("Observed vs Fitted TEST set")
plt.tight_layout()
plt.show()

#### BMI as category

In [0]:
#DME
#drop winter 
model_305_cat = ols('logPersistency ~ C(DME_prop_cat) + C(LactationCat) + CalvingYear + CalvingSeason_Spring + CalvingSeason_Summer + CalvingSeason_Autumn', data=train_df)
model_305_cat = model_305_cat.fit()
print(model_305_cat.summary())

#visualize results
y = train_df['logPersistency'].values
yhat = model_305_cat.fittedvalues

plt.figure(figsize=(6,6))
plt.scatter(y, yhat, alpha=0.5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], linewidth=2)  # 1:1 line

plt.xlabel("Observed logPersistency")
plt.ylabel("Fitted logPersistency")
plt.title("Observed vs Fitted")
plt.tight_layout()
plt.show()

print('Train RMSE: ', np.sqrt(mean_squared_error(y, yhat)))

##### Eval test set

In [0]:
#Evaluate using the testset 
#center calvingyear using the mean of the train set 
# test_df['CalvingYear'] = test_df['CalvingYear'] - mean_calving_year

y_test = test_df['logPersistency'].values
ypred_test = model_305_cat.predict(test_df)
print('Test R2: ', r2_score(y_test, ypred_test))
print('Test RMSE: ', np.sqrt(mean_squared_error(y_test, ypred_test)))

#visualize results

plt.figure(figsize=(6,6))
plt.scatter(y_test, ypred_test, alpha=0.5, color = 'darkorange')
plt.plot([y.min(), y.max()], [y.min(), y.max()], linewidth=2, color = 'darkorange')  # 1:1 line

plt.xlabel("Observed logPersistency")
plt.ylabel("Fitted logPersistency")
plt.title("Observed vs Fitted TEST set")
plt.tight_layout()
plt.show()

In [0]:
# fit linear regression model 

#drop winter 
model_305 = ols('CummulativeYield ~ C(DME_prop_cat) + C(LactationCat) + CalvingYear + CalvingSeason_Spring + CalvingSeason_Summer + CalvingSeason_Autumn', data=train_df)
model_305 = model_305.fit()
print(model_305.summary())

#visualize results
y = train_df['CummulativeYield'].values
yhat = model_305.fittedvalues

plt.figure(figsize=(6,6))
plt.scatter(y, yhat, alpha=0.5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], linewidth=2)  # 1:1 line

plt.xlabel("Observed Cummulative milk yield")
plt.ylabel("Fitted Cummulative milk yield")
plt.title("Observed vs Fitted")
plt.tight_layout()
plt.show()

print('Train RMSE: ', np.sqrt(mean_squared_error(y, yhat)))

In [0]:
y_test = test_df['CummulativeYield'].values
ypred_test = model_305.predict(test_df)
print('Test R2: ', r2_score(y_test, ypred_test))
print('Test RMSE: ', np.sqrt(mean_squared_error(y_test, ypred_test)))

#visualize results

plt.figure(figsize=(6,6))
plt.scatter(y_test, ypred_test, alpha=0.5, color = 'darkorange')
plt.plot([y.min(), y.max()], [y.min(), y.max()], linewidth=2, color = 'darkorange')  # 1:1 line

plt.xlabel("Observed Cummulative milk yield")
plt.ylabel("Fitted Cummulative milk yield")
plt.title("Observed vs Fitted TEST set")
plt.tight_layout()
plt.show()

In [0]:
grouped = pdf.groupby("DME_prop_cat")["Persistency"]

mean = grouped.mean()
sem = grouped.sem()                 # standard error of the mean
ci95 = 1.96 * sem                   # 95% CI

plt.figure(figsize=(6,4))
plt.bar(mean.index, mean.values, yerr=ci95.values, capsize=5)
plt.ylabel("Mean persistency (kg/day)")
plt.xlabel("DME proportion category")
plt.title("Persistency by DME proportion category\n(Mean ± 95% CI)")
plt.tight_layout()
plt.show()


In [0]:
grouped = pdf.groupby("DME_prop_cat")["Persistency"]

mean = grouped.mean()
sem = grouped.sem()
ci95 = 1.96 * sem

# -----------------------------
# Broken y-axis configuration
# -----------------------------
focus_ylim = (-0.08, -0.06)   # most negative (important)
context_ylim = (-0.06, 0)     # less negative (context)

fig = plt.figure(figsize=(6, 5))

gs = fig.add_gridspec(
    2, 1,
    height_ratios=[3, 1],   # 70% focus region
    hspace=0.05
)

ax_focus = fig.add_subplot(gs[0])
ax_context = fig.add_subplot(gs[1], sharex=ax_focus)

# -----------------------------
# Plot bars
# -----------------------------
for ax in (ax_focus, ax_context):
    ax.bar(mean.index, mean.values, yerr=ci95.values, capsize=5)

# -----------------------------
# Set y-limits
# -----------------------------
ax_focus.set_ylim(focus_ylim)
ax_context.set_ylim(context_ylim)

# 🔑 Key fix: invert y-axes
ax_focus.invert_yaxis()
ax_context.invert_yaxis()

# -----------------------------
# Remove spines between axes
# -----------------------------
ax_focus.spines['bottom'].set_visible(False)
ax_context.spines['top'].set_visible(False)
ax_focus.tick_params(labelbottom=False)

# -----------------------------
# Diagonal break marks
# -----------------------------
d = .015
kwargs = dict(transform=ax_focus.transAxes, color='k', clip_on=False)
ax_focus.plot((-d, +d), (1 - d, 1 + d), **kwargs)
ax_focus.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)

kwargs.update(transform=ax_context.transAxes)
ax_context.plot((-d, +d), (-d, +d), **kwargs)
ax_context.plot((1 - d, 1 + d), (-d, +d), **kwargs)

# -----------------------------
# Labels and title
# -----------------------------
ax_context.set_xlabel("DME proportion category")
ax_focus.set_ylabel("Mean persistency (kg/day)")
fig.suptitle(
    "Persistency by DME proportion category\n(Mean ± 95% CI)",
    y=0.96
)

plt.tight_layout()
plt.show()


In [0]:
plt.figure(figsize=(6,4))
pdf.boxplot(
    column="logPersistency",
    by="DME_prop_cat",
    grid=False
)
plt.ylabel("logPersistency (kg/day)")
plt.xlabel("DME proportion category")
plt.title("Distribution of logPersistency by DME category")
plt.suptitle("")  # remove automatic pandas title
plt.tight_layout()
plt.show()

#### Two step approach 

In [0]:
response = "logPersistency"

predictors = [
    "DME_prop_cat",
    "C(LactationCat)",
    "CalvingYear",
    "CalvingSeason_Spring",
    "CalvingSeason_Summer",
    "CalvingSeason_Autumn"
]

#fit a model with only an intercept
m0 = ols(f"{response} ~ 1", data=train_df).fit()

print("M0 (Intercept only)")
print(f"AIC: {m0.aic:.2f}")
print(f"BIC: {m0.bic:.2f}")

#fit one predictor models
results = []

for pred in predictors:
    formula = f"{response} ~ {pred}"
    model = ols(formula, data=train_df).fit()

    results.append({
        "predictor": pred,
        "AIC": model.aic,
        "BIC": model.bic,
        "delta_AIC": model.aic - m0.aic,
        "delta_BIC": model.bic - m0.bic
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("AIC")
    .reset_index(drop=True)
)

print(results_df)

#Select predictors that improve AIC, threshold <-2
selected_predictors = results_df.loc[
    results_df["delta_AIC"] < -2,
    "predictor"
].tolist()

print("Selected predictors:")
print(selected_predictors)

#create final model 
final_formula = (
    f"{response} ~ " + " + ".join(selected_predictors)
)

current_model = ols(final_formula, data=train_df).fit()

print(current_model.summary())
print(f"Final AIC: {current_model.aic:.2f}")
print(f"Final BIC: {current_model.bic:.2f}")

#visualize results
y = train_df['logPersistency'].values
yhat = current_model.fittedvalues

plt.figure(figsize=(6,6))
plt.scatter(y, yhat, alpha=0.5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], linewidth=2)  # 1:1 line

plt.xlabel("Observed logPersistency yield")
plt.ylabel("Fitted logPersistency")
plt.title("Observed vs Fitted")
plt.tight_layout()
plt.show()

print('Train RMSE: ', np.sqrt(mean_squared_error(y, yhat)))

In [0]:
history = []

current_formula = final_formula
current_aic = current_model.aic

step = 0

while True:
    step += 1

    # Extract p-values (drop intercept)
    pvals = current_model.pvalues.drop("Intercept")

    # Stop if everything is significant
    if (pvals < 0.05).all():
        print("All remaining predictors are significant.")
        break

    # Predictor with largest p-value
    worst_pred = pvals.idxmax()
    
    # Extract base predictor name (before '[' for categorical variables)
    worst_pred_base = worst_pred.split('[')[0]

    # Build new formula without that predictor
    terms = current_formula.split("~")[1].strip().split(" + ")
    terms.remove(worst_pred_base)

    new_formula = "logPersistency~ " + " + ".join(terms)

    new_model = ols(new_formula, data=train_df).fit()

    history.append({
        "step": step,
        "dropped": worst_pred,
        "AIC": new_model.aic,
        "delta_AIC": new_model.aic - current_aic
    })

    # Stop if AIC gets worse
    if new_model.aic > current_aic:
        print(f"Dropping {worst_pred} worsened AIC — stopping.")
        break

    # Accept the new model
    current_formula = new_formula
    current_model = new_model
    current_aic = new_model.aic

history_df = pd.DataFrame(history)
print(history_df)

In [0]:
#continuous variable DME

response = "logPersistency"

predictors = [
    "DME_prop_logit",
    "C(LactationCat)",
    "CalvingYear",
    "CalvingSeason_Spring",
    "CalvingSeason_Summer",
    "CalvingSeason_Autumn"
]

#fit a model with only an intercept
m0 = ols(f"{response} ~ 1", data=train_df).fit()

print("M0 (Intercept only)")
print(f"AIC: {m0.aic:.2f}")
print(f"BIC: {m0.bic:.2f}")

#fit one predictor models
results = []

for pred in predictors:
    formula = f"{response} ~ {pred}"
    model = ols(formula, data=train_df).fit()

    results.append({
        "predictor": pred,
        "AIC": model.aic,
        "BIC": model.bic,
        "delta_AIC": model.aic - m0.aic,
        "delta_BIC": model.bic - m0.bic
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("AIC")
    .reset_index(drop=True)
)

print(results_df)

#Select predictors that improve AIC, threshold <-2
selected_predictors = results_df.loc[
    results_df["delta_AIC"] < -2,
    "predictor"
].tolist()

print("Selected predictors:")
print(selected_predictors)

#create final model 
final_formula = (
    f"{response} ~ " + " + ".join(selected_predictors)
)

current_model = ols(final_formula, data=train_df).fit()

print(current_model.summary())
print(f"Final AIC: {current_model.aic:.2f}")
print(f"Final BIC: {current_model.bic:.2f}")

#visualize results
y = train_df['logPersistency'].values
yhat = current_model.fittedvalues

plt.figure(figsize=(6,6))
plt.scatter(y, yhat, alpha=0.5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], linewidth=2)  # 1:1 line

plt.xlabel("Observed logPersistency yield")
plt.ylabel("Fitted logPersistency")
plt.title("Observed vs Fitted")
plt.tight_layout()
plt.show()

print('Train RMSE: ', np.sqrt(mean_squared_error(y, yhat)))

In [0]:
history = []

current_formula = final_formula
current_aic = current_model.aic

step = 0

while True:
    step += 1

    # Extract p-values (drop intercept)
    pvals = current_model.pvalues.drop("Intercept")

    # Stop if everything is significant
    if (pvals < 0.05).all():
        print("All remaining predictors are significant.")
        break

    # Predictor with largest p-value
    worst_pred = pvals.idxmax()
    


    # Build new formula without that predictor
    terms = current_formula.split("~")[1].strip().split(" + ")
    terms.remove(worst_pred)

    new_formula = "logPersistency~ " + " + ".join(terms)

    new_model = ols(new_formula, data=train_df).fit()

    history.append({
        "step": step,
        "dropped": worst_pred,
        "AIC": new_model.aic,
        "delta_AIC": new_model.aic - current_aic
    })

    # Stop if AIC gets worse
    if new_model.aic > current_aic:
        print(f"Dropping {worst_pred} worsened AIC — stopping.")
        break

    # Accept the new model
    current_formula = new_formula
    current_model = new_model
    current_aic = new_model.aic

history_df = pd.DataFrame(history)
print(history_df)

## Random Forest

## Random forrest with normal DME prop

In [0]:
#redefine input and output variables 
X_train = train_df[['DME_prop', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn']]
y_train = train_df['logPersistency']
X_test = test_df[['DME_prop', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn']]
y_test = test_df['logPersistency']

#define parameter grid 
param_grid = {
    "n_estimators": [250, 500, 1000],
    "max_depth": [3, 5, 7],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 3, 5]
    }

#define model
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=10,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
print("Best parameters:", grid_search.best_params_)
best_rf_model = grid_search.best_estimator_

# Predictions
y_pred_train = best_rf_model.predict(X_train)

#calculate training metrics 
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
r_squared_train = r2_score(y_train, y_pred_train)
print('Training RMSE: ', rmse_train)
print('Training R2: ', r_squared_train)

# #feature importance 
# feature_names = ['DME_prop', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn']

# importances = pd.Series(best_rf_model.feature_importances_, index=feature_names)
# importances.sort_values().plot.barh(title = 'Feature importance RF model peak yield')
# plt.show()


In [0]:
y_pred_test = best_rf_model.predict(X_test)

#calculate testing metrics 
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
r_squared_test = r2_score(y_test, y_pred_test)
print('non logit')
print('Testing RMSE: ', rmse_test)
print('Testing R2: ', r_squared_test)

## XgBoost model with hyperparameter tuning

## XgBoost non logit

In [0]:
#define input and output variables 
X_train = train_df[['DME_prop', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn']]
y_train = train_df['logPersistency']
X_test = test_df[['DME_prop', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn']]
y_test = test_df['logPersistency']


#Grid search parameters
param_grid = {
    "n_estimators": [250, 500, 1000],
    "max_depth": [1, 3, 5, 7],
    "learning_rate": [0.001, 0.01, 0.1],
    "min_child_weight": [1, 3, 5]
    }

    
#define model
grid_search = GridSearchCV(
        estimator=XGBRegressor(objective='reg:squarederror', random_state=42),
        param_grid=param_grid,
        cv=10,
        n_jobs=-1
        )

 # Fit model
grid_search.fit(X_train, y_train) 
print("Best parameters:", grid_search.best_params_)

best_model = grid_search.best_estimator_

# Predictions
y_pred = best_model.predict(X_train)

# RMSE
rmse = np.sqrt(mean_squared_error(y_train, y_pred))
print(f"RMSE: {rmse:.3f}")

r2 = r2_score(y_train, y_pred)
print(f"R²: {r2:.3f}")

# # Feature importance 
# importance_XgBoost = pd.Series(best_model.feature_importances_, index=feature_names)
# importance_XgBoost = importance_XgBoost.sort_values(ascending=True)

# importance_XgBoost.plot.barh(figsize=(6,4))
# plt.title(f"Feature Importance XgBoost peak yield")
# plt.show()

In [0]:
y_pred_test = best_model.predict(X_test)

#calculate testing metrics 
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
r_squared_test = r2_score(y_test, y_pred_test)
print('non logit')
print('Testing RMSE: ', rmse_test)
print('Testing R2: ', r_squared_test)

### Permutation feature importance 

In [0]:
#recreate best fitted models for RF and XgBoost 
#define input and output variables 
X_train = train_df[['DME_prop_logit', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn', 'CalvingSeason_Winter']]
y_train = train_df['logPersistency']
X_test = test_df[['DME_prop_logit', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn', 'CalvingSeason_Winter']]
y_test = test_df['logPersistency']

#fit model 
RF = RandomForestRegressor(n_estimators=1000, max_depth=5, min_samples_leaf=1, min_samples_split=10, random_state=42).fit(X_train, y_train)
XgBoost = XGBRegressor(objective='reg:squarederror', random_state=42, n_estimators = 1000, learning_rate = 0.01, min_child_weight = 1, max_depth = 1).fit(X_train, y_train)

# Predictions
rf_y_pred_train = RF.predict(X_train)
x_y_pred_train = XgBoost.predict(X_train)

#calculate training metrics 
rmse_train = np.sqrt(mean_squared_error(y_train, rf_y_pred_train))
r_squared_train = r2_score(y_train, rf_y_pred_train)
print('Training RMSE RF: ', rmse_train)
print('Training R2 RF: ', r_squared_train)
rmse_train = np.sqrt(mean_squared_error(y_train, x_y_pred_train))
r_squared_train = r2_score(y_train, x_y_pred_train)
print('Training RMSE XgBoost: ', rmse_train)
print('Training R2 XgBoost: ', r_squared_train)


from sklearn.inspection import permutation_importance

# ---- Permutation Importance for Random Forest ----
rf_perm = permutation_importance(
    RF, 
    X_test, 
    y_test, 
    n_repeats=30,     
    random_state=42, 
    scoring='r2'
)

rf_importance = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': rf_perm.importances_mean,
    'importance_std': rf_perm.importances_std
}).sort_values(by='importance_mean', ascending=False)

print("Random Forest - Permutation Importance")
print(rf_importance)


# ---- Permutation Importance for XGBoost ----
xgb_perm = permutation_importance(
    XgBoost, 
    X_test, 
    y_test, 
    n_repeats=30, 
    random_state=42, 
    scoring='r2'
)

xgb_importance = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': xgb_perm.importances_mean,
    'importance_std': xgb_perm.importances_std
}).sort_values(by='importance_mean', ascending=False)

print("\nXGBoost - Permutation Importance")
print(xgb_importance)



In [0]:
print(f'relative importance lactationcat vs DME RF {0.006232  /0.769150 }')
print(f'relative importance lactationcat vs DME XgBoost {0.006502/0.743735}')

In [0]:
def plot_perm_importance(df, title):

    #rename feature labels
    df['feature'] = df['feature'].replace({'DME_prop_logit': 'DME', 'LactationCat': 'Parity', 'CalvingYear': 'Calving Year', 'CalvingSeason_Spring': 'Spring', 'CalvingSeason_Summer': 'Summer', 'CalvingSeason_Autumn': 'Autumn', 'CalvingSeason_Winter': 'Winter'})

    plt.figure()
    plt.barh(df['feature'], df['importance_mean'])
    plt.xlabel("Permutation Importance (mean ΔR²)")
    plt.title(title)
    plt.gca().invert_yaxis()
    plt.xlim(left=0, right = 1)  # Ensure x-axis starts at 0
    plt.show()

plot_perm_importance(rf_importance, "Random forest permutation importance persistency")
plot_perm_importance(xgb_importance, "XGBoost permutation importance persistency")

In [0]:
#without title
plot_perm_importance(rf_importance, "")
plot_perm_importance(xgb_importance, "")

Feature importance of non log transformed persistency (without hyperparameter tuning)

In [0]:
#recreate best fitted models for RF and XgBoost 
#define input and output variables 
X_train = train_df[['DME_prop_logit', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn', 'CalvingSeason_Winter']]
y_train = train_df['Persistency']
X_test = test_df[['DME_prop_logit', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn', 'CalvingSeason_Winter']]
y_test = test_df['Persistency']

#fit model 
RF = RandomForestRegressor(n_estimators=1000, max_depth=5, min_samples_leaf=1, min_samples_split=10, random_state=42).fit(X_train, y_train)
XgBoost = XGBRegressor(objective='reg:squarederror', random_state=42, n_estimators = 1000, learning_rate = 0.01, min_child_weight = 1, max_depth = 1).fit(X_train, y_train)

# Predictions
rf_y_pred_train = RF.predict(X_train)
x_y_pred_train = XgBoost.predict(X_train)

#calculate training metrics 
rmse_train = np.sqrt(mean_squared_error(y_train, rf_y_pred_train))
r_squared_train = r2_score(y_train, rf_y_pred_train)
print('Training RMSE RF: ', rmse_train)
print('Training R2 RF: ', r_squared_train)
rmse_train = np.sqrt(mean_squared_error(y_train, x_y_pred_train))
r_squared_train = r2_score(y_train, x_y_pred_train)
print('Training RMSE XgBoost: ', rmse_train)
print('Training R2 XgBoost: ', r_squared_train)


from sklearn.inspection import permutation_importance

# ---- Permutation Importance for Random Forest ----
rf_perm = permutation_importance(
    RF, 
    X_test, 
    y_test, 
    n_repeats=30,     
    random_state=42, 
    scoring='r2'
)

rf_importance = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': rf_perm.importances_mean,
    'importance_std': rf_perm.importances_std
}).sort_values(by='importance_mean', ascending=False)

print("Random Forest - Permutation Importance")
print(rf_importance)


# ---- Permutation Importance for XGBoost ----
xgb_perm = permutation_importance(
    XgBoost, 
    X_test, 
    y_test, 
    n_repeats=30, 
    random_state=42, 
    scoring='r2'
)

xgb_importance = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': xgb_perm.importances_mean,
    'importance_std': xgb_perm.importances_std
}).sort_values(by='importance_mean', ascending=False)

print("\nXGBoost - Permutation Importance")
print(xgb_importance)

plot_perm_importance(rf_importance, "Random Forest Permutation Importance persistency")
plot_perm_importance(xgb_importance, "XGBoost Permutation Importance persistency")


## SHAP

In [0]:
#recreate best fitted models for RF and XgBoost 

#define input and output variables 
X_train = train_df[['DME_prop_logit', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn', 'CalvingSeason_Winter']]
y_train = train_df['Persistency']

#change columnnames
X_train = X_train.rename(columns={'DME_prop_logit': 'DME', 'LactationCat': 'Parity', 'CalvingYear': 'Calving Year', 'CalvingSeason_Spring': 'Spring', 'CalvingSeason_Summer': 'Summer', 'CalvingSeason_Autumn': 'Autumn', 'CalvingSeason_Winter': 'Winter'})

#fit model 
RF = RandomForestRegressor(n_estimators=1000, max_depth=5, min_samples_leaf=1, min_samples_split=10, random_state=42).fit(X_train, y_train)
XgBoost = XGBRegressor(objective='reg:squarederror', random_state=42, n_estimators = 1000, learning_rate = 0.01, min_child_weight = 1, max_depth = 1).fit(X_train, y_train)

### Random forest

In [0]:
import shap
# get shap values
explainer = shap.Explainer(RF)
shap_values = explainer(X_train)

#sum up season category
calving_season_cols = [
    'Spring',
    'Summer',
    'Autumn',
    'Winter'
]

# indices
season_idx = [shap_values.feature_names.index(col) for col in calving_season_cols]

# summed shap values
calving_season_sum = shap_values.values[:, season_idx].sum(axis=1)

# remaining columns
remaining_idx = [
    i for i, col in enumerate(shap_values.feature_names)
    if col not in calving_season_cols
]

# reduced arrays
new_values = shap_values.values[:, remaining_idx]
new_data = shap_values.data[:, remaining_idx]

# insertion position
insert_pos = remaining_idx.index(
    shap_values.feature_names.index('Calving Year')
) + 1

# insert summed SHAP values
new_values = np.insert(
    new_values,
    insert_pos,
    calving_season_sum,
    axis=1
)

# create numeric category encoding
season_numeric = shap_values.data[:, season_idx].argmax(axis=1)

# insert numeric values into data
new_data = np.insert(
    new_data,
    insert_pos,
    season_numeric,
    axis=1
)

# feature names
new_feature_names = [
    shap_values.feature_names[i]
    for i in remaining_idx
]
new_feature_names.insert(insert_pos, "Calving Season")

# create new explanation
shap_values_new = shap.Explanation(
    values=new_values,
    base_values=shap_values.base_values,
    data=new_data,
    feature_names=new_feature_names
)

#visualize 

print('Random Forest SHAP')
print('Waterfall plots')
# waterfall plot for sample of observation
shap.plots.waterfall(shap_values_new[5220])
shap.plots.waterfall(shap_values_new[0])
shap.plots.waterfall(shap_values_new[5219])
shap.plots.waterfall(shap_values_new[3])

print('')
shap.plots.bar(shap_values_new)
shap.plots.beeswarm(shap_values_new)
shap.plots.scatter(shap_values_new[:, 'DME'], color = shap_values_new)

### XgBoost

In [0]:
explainer_xgboost = shap.Explainer(XgBoost)
shap_values_xgboost = explainer_xgboost(X_train)

calving_season_cols = [
    'Spring',
    'Summer',
    'Autumn',
    'Winter'
]

# indices
season_idx = [shap_values_xgboost.feature_names.index(col) for col in calving_season_cols]

# summed shap values
calving_season_sum = shap_values_xgboost.values[:, season_idx].sum(axis=1)

# remaining columns
remaining_idx = [
    i for i, col in enumerate(shap_values_xgboost.feature_names)
    if col not in calving_season_cols
]

# reduced arrays
new_values = shap_values_xgboost.values[:, remaining_idx]
new_data = shap_values_xgboost.data[:, remaining_idx]

# insertion position
insert_pos = remaining_idx.index(
    shap_values_xgboost.feature_names.index('Calving Year')
) + 1

# insert summed SHAP values
new_values = np.insert(
    new_values,
    insert_pos,
    calving_season_sum,
    axis=1
)

# create numeric category encoding
season_numeric = shap_values_xgboost.data[:, season_idx].argmax(axis=1)

# insert numeric values into data
new_data = np.insert(
    new_data,
    insert_pos,
    season_numeric,
    axis=1
)

# feature names
new_feature_names = [
    shap_values_xgboost.feature_names[i]
    for i in remaining_idx
]
new_feature_names.insert(insert_pos, "Calving Season")

# create new explanation
shap_values_xgboost_new = shap.Explanation(
    values=new_values,
    base_values=shap_values_xgboost.base_values,
    data=new_data,
    feature_names=new_feature_names
)

# waterfall plot for sample of observation
print('XGBoost SHAP')
print('Waterfall plots')
shap.plots.waterfall(shap_values_xgboost_new[5220])
shap.plots.waterfall(shap_values_xgboost_new[0])
shap.plots.waterfall(shap_values_xgboost_new[5219])
shap.plots.waterfall(shap_values_xgboost_new[3])

print('')
shap.plots.bar(shap_values_xgboost_new)
shap.plots.beeswarm(shap_values_xgboost_new)
shap.plots.scatter(shap_values_xgboost_new[:, 'DME'], color = shap_values_xgboost_new)